# Physics-Informed TiDE (PI-TiDE) for Electricity Demand Forecasting

This is an **original implementation** assembled for this project — it is not a transcription of a
published "PI-TiDE" paper (none was found in search at time of writing). It combines two things that
are each independently established:

1. **TiDE architecture** — Das et al., *"Long-term Forecasting with TiDE: Time-series Dense Encoder"*
   (2023, arXiv:2304.08424): residual-block MLP encoder/decoder with a global linear residual and a
   temporal decoder that mixes decoded features with future covariates.
2. **Physics-informed loss** — in the spirit of PINNs (Raissi et al., 2019) applied to load
   forecasting: penalizing predictions whose local sensitivity to temperature has the wrong sign
   relative to the known heating/cooling degree-day (HDD/CDD) regime, plus non-negativity and
   ramp-rate physical constraints.

**Treat the physics-loss design as a hypothesis to validate empirically** — ablate the lambda
weights, check whether the physical-violation-rate metric drops, and confirm MAPE/RMSE don't
regress. This is not a settled, peer-reviewed recipe. Cross-check the TiDE architectural details
against the original paper or Darts' own `TiDEModel` source if exact reproduction matters:
https://github.com/unit8co/darts

**Expected input dataframe columns** (rename to match your dataset):
- `demand` — target load (BZN|GR or Bangladesh PGCB series)
- `temp` — temperature covariate (known/forecast for the future window)
- `humidity`, `is_holiday` — optional additional covariates

Runs on CPU or GPU (Kaggle/Colab T4, P100) automatically.

## 0. Setup

In [1]:
!pip install -q torch numpy pandas scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


## 1. Physical prior: heating/cooling degree-day regime

Encodes the well-known piecewise temperature-demand relationship: below the balance temperature
`T_b`, demand rises as temperature **falls** (heating); above `T_b`, demand rises as temperature
**rises** (cooling). This is a fixed physical prior, not a learned component.

`balance_temp` should be estimated from **your** data (see the helper in Section 5) rather than
assumed — 22°C is a placeholder, not a universal constant. Verify against your own
load-vs-temperature scatter plot.

In [3]:
class DegreeDayPhysics:
    def __init__(self, balance_temp: float = 22.0, cooling_only: bool = False):
        self.T_b = balance_temp
        # Bangladesh data is monotonic rising (no heating regime): fix expected sign to +1
        self.cooling_only = cooling_only
        # Temperature-power envelope (populated by fit_envelope)
        self.envelope_temps = None
        self.envelope_max_power = None

    def hdd(self, temp: torch.Tensor) -> torch.Tensor:
        return torch.clamp(self.T_b - temp, min=0.0)

    def cdd(self, temp: torch.Tensor) -> torch.Tensor:
        return torch.clamp(temp - self.T_b, min=0.0)

    def expected_sensitivity_sign(self, temp: torch.Tensor) -> torch.Tensor:
        """Expected sign of d(demand)/d(temp): -1 heating regime, +1 cooling regime.
        cooling_only=True forces +1 everywhere (no heating regime) -- use for monotonic
        load-vs-temperature data where demand rises even in the cold season."""
        if self.cooling_only:
            return torch.ones_like(temp)
        return torch.where(temp < self.T_b, -torch.ones_like(temp), torch.ones_like(temp))

    def fit_envelope(self, df, temp_col="temp", target_col="demand", n_bins=30):
        """Fit piecewise-linear upper envelope P_max(T) from data (binned max demand)."""
        lo, hi = df[temp_col].min(), df[temp_col].max()
        bins = np.linspace(lo, hi, n_bins + 1)
        df = df.copy()
        df["tbin"] = pd.cut(df[temp_col], bins=bins)
        max_power = df.groupby("tbin", observed=True)[target_col].max()
        # Forward fill NaNs, then interpolate
        max_power = max_power.interpolate().bfill().ffill()
        self.envelope_temps = np.array([b.mid for b in max_power.index])
        self.envelope_max_power = max_power.values.astype(np.float32)

    def max_power_at_temp(self, temp: torch.Tensor) -> torch.Tensor:
        """Interpolate envelope at given temperatures."""
        if self.envelope_temps is None:
            return torch.full_like(temp, float("inf"))
        # numpy interp on CPU, then back to tensor
        temp_np = temp.detach().cpu().numpy().ravel()
        interp = np.interp(temp_np, self.envelope_temps, self.envelope_max_power)
        return torch.from_numpy(interp).to(temp.device).reshape(temp.shape)


def estimate_balance_temp_piecewise(df, target_col, temp_col):
    """Fit segmented regression: demand = a1*T + b1 (T<Tb) + a2*T + b2 (T>=Tb)."""
    from scipy.optimize import minimize_scalar
    from sklearn.linear_model import LinearRegression
    def rss(tb):
        cold = df[df[temp_col] < tb]
        hot = df[df[temp_col] >= tb]
        if len(cold) < 10 or len(hot) < 10:
            return np.inf
        m1 = LinearRegression().fit(cold[[temp_col]], cold[target_col])
        m2 = LinearRegression().fit(hot[[temp_col]], hot[target_col])
        rss1 = np.sum((cold[target_col] - m1.predict(cold[[temp_col]]))**2)
        rss2 = np.sum((hot[target_col] - m2.predict(hot[[temp_col]]))**2)
        return rss1 + rss2
    lo, hi = df[temp_col].quantile(0.1), df[temp_col].quantile(0.9)
    res = minimize_scalar(rss, bounds=(lo, hi), method='bounded')
    return float(res.x)


## 2. TiDE building blocks

In [4]:
class ResidualBlock(nn.Module):
    """Linear -> ReLU -> Linear -> Dropout, added to a (projected) skip connection, then (optional) LayerNorm."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.1, use_layer_norm: bool = True):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()
        self.norm = nn.LayerNorm(output_dim) if use_layer_norm else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.fc1(x))
        h = self.fc2(h)
        h = self.dropout(h)
        return self.norm(h + self.skip(x))


class ResidualStack(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, dropout=0.1, use_layer_norm=True):
        super().__init__()
        dims = [input_dim] + [hidden_dim] * (n_layers - 1) + [output_dim]
        self.blocks = nn.ModuleList(
            [ResidualBlock(dims[i], hidden_dim, dims[i + 1], dropout, use_layer_norm) for i in range(n_layers)]
        )

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


In [5]:
class PITiDE(nn.Module):
    """
    Physics-informed TiDE.

    lookback      : length of input target window (L)
    horizon       : forecast length (H)
    n_covariates  : number of time-varying covariates (temp, humidity, is_holiday, calendar...)
    temp_idx      : index of the temperature covariate within the covariate feature vector
                    (needed so the physics loss can find it and take gradients w.r.t. it)
    hidden_dim / hiddenSize           : width of encoder/decoder residual blocks
    encoder_layers / numEncoderLayers : depth of encoder stack
    decoder_layers / numDecoderLayers : depth of decoder stack
    decoder_output_dim                : per-horizon-step decoded feature width
    temporal_decoder_hidden           : hidden width inside the temporal decoder
    dropout / dropoutLevel            : dropout rate
    use_layer_norm / layerNorm        : True -> LayerNorm after each residual add, False -> Identity
    use_revin / revIn                 : True -> Reversible Instance Norm on the target window
    """

    def __init__(
        self,
        lookback: int,
        horizon: int,
        n_covariates: int,
        temp_idx: int,
        hidden_dim: int = 128,
        feature_proj_dim: int = 8,
        encoder_layers: int = 2,
        decoder_layers: int = 2,
        decoder_output_dim: int = 16,
        temporal_decoder_hidden: int = None,  # None -> fall back to hidden_dim (backward-compat)
        dropout: float = 0.1,
        use_layer_norm: bool = True,
        use_revin: bool = False,
    ):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.n_covariates = n_covariates
        self.temp_idx = temp_idx
        self.use_revin = use_revin
        if temporal_decoder_hidden is None:
            temporal_decoder_hidden = hidden_dim

        self.feature_projection = ResidualBlock(n_covariates, hidden_dim, feature_proj_dim, dropout, use_layer_norm)

        encoder_input_dim = lookback + (lookback + horizon) * feature_proj_dim
        self.encoder = ResidualStack(encoder_input_dim, hidden_dim, hidden_dim, encoder_layers, dropout, use_layer_norm)

        self.decoder = ResidualStack(
            hidden_dim, hidden_dim, horizon * decoder_output_dim, decoder_layers, dropout, use_layer_norm
        )
        self.decoder_output_dim = decoder_output_dim

        self.temporal_decoder = ResidualBlock(
            decoder_output_dim + feature_proj_dim, temporal_decoder_hidden, 1, dropout, use_layer_norm
        )

        self.global_residual = nn.Linear(lookback, horizon)

    def forward(self, past_target: torch.Tensor, covariates: torch.Tensor):
        """
        past_target : (batch, lookback)
        covariates  : (batch, lookback + horizon, n_covariates)
        returns     : predictions (batch, horizon)
        """
        batch_size = past_target.shape[0]

        if self.use_revin:
            # RevIN (Kim et al., 2021): per-series instance norm on the lookback, denorm on output.
            rev_mean = past_target.mean(dim=1, keepdim=True)
            rev_std = past_target.std(dim=1, keepdim=True).clamp_min(1e-5)
            past_target_in = (past_target - rev_mean) / rev_std
        else:
            past_target_in = past_target

        proj = self.feature_projection(covariates)
        proj_flat = proj.reshape(batch_size, -1)

        enc_input = torch.cat([past_target_in, proj_flat], dim=-1)
        encoded = self.encoder(enc_input)

        decoded = self.decoder(encoded).reshape(batch_size, self.horizon, self.decoder_output_dim)

        future_proj = proj[:, self.lookback:, :]
        temporal_input = torch.cat([decoded, future_proj], dim=-1)
        residual_out = self.temporal_decoder(temporal_input).squeeze(-1)

        global_out = self.global_residual(past_target_in)

        out = residual_out + global_out
        if self.use_revin:
            out = out * rev_std + rev_mean
        return out


## 3. Physics-informed loss

Three differentiable penalty terms:
- **(a)** temperature-sensitivity sign consistency, via `torch.autograd.grad` of the prediction
  w.r.t. the temperature covariate
- **(b)** non-negativity (demand can't be physically negative)
- **(c)** ramp-rate bound (99th-percentile-derived)

In [6]:
def physics_informed_loss(
    model,
    past_target: torch.Tensor,
    covariates: torch.Tensor,
    predictions: torch.Tensor,
    physics: DegreeDayPhysics,
    max_ramp: float,
    lambda_sens: float = 0.1,
    lambda_nonneg: float = 0.1,
    lambda_ramp: float = 0.1,
    lambda_env: float = 0.1,      # temperature-power envelope penalty
):
    """
    Returns (total_physics_loss, components_dict). Requires `covariates` to have
    requires_grad=True (set in the training loop) so we can differentiate the
    prediction w.r.t. the temperature channel.
    """
    future_temp = covariates[:, model.lookback:, model.temp_idx]

    grad_outputs = torch.ones_like(predictions)
    d_pred_d_cov = torch.autograd.grad(
        outputs=predictions,
        inputs=covariates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
    )[0]
    d_pred_d_temp = d_pred_d_cov[:, model.lookback:, model.temp_idx]

    expected_sign = physics.expected_sensitivity_sign(future_temp)
    sensitivity_violation = F.relu(-expected_sign * d_pred_d_temp)
    sensitivity_loss = sensitivity_violation.mean()

    nonneg_loss = F.relu(-predictions).pow(2).mean()

    diffs = predictions[:, 1:] - predictions[:, :-1]
    ramp_violation = F.relu(diffs.abs() - max_ramp)
    ramp_loss = ramp_violation.pow(2).mean()

    # --- Temperature-power envelope: demand <= P_max(T) ---
    # physics.max_power_at_temp must be implemented (piecewise linear from data)
    if hasattr(physics, "max_power_at_temp"):
        max_power = physics.max_power_at_temp(future_temp)
        envelope_violation = F.relu(predictions - max_power).pow(2).mean()
    else:
        envelope_violation = torch.tensor(0.0, device=predictions.device)

    total = (lambda_sens * sensitivity_loss +
             lambda_nonneg * nonneg_loss +
             lambda_ramp * ramp_loss +
             lambda_env * envelope_violation)
    components = {
        "sensitivity_loss": sensitivity_loss.item(),
        "nonneg_loss": nonneg_loss.item(),
        "ramp_loss": ramp_loss.item(),
        "envelope_loss": envelope_violation.item(),
    }
    return total, components


## 4. Dataset: sliding windows over demand + covariates

In [7]:
class DemandWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, target_col: str, covariate_cols: list, lookback: int, horizon: int):
        self.target = df[target_col].values.astype(np.float32)
        self.covariates = df[covariate_cols].values.astype(np.float32)
        self.lookback = lookback
        self.horizon = horizon
        self.n = len(df) - lookback - horizon + 1
        if self.n <= 0:
            raise ValueError("Dataframe too short for given lookback/horizon.")

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        past_target = self.target[idx: idx + self.lookback]
        cov_window = self.covariates[idx: idx + self.lookback + self.horizon]
        future_target = self.target[idx + self.lookback: idx + self.lookback + self.horizon]
        return (
            torch.from_numpy(past_target),
            torch.from_numpy(cov_window),
            torch.from_numpy(future_target),
        )

## 5. Helpers: chronological split, balance-temp estimation, real-unit metrics

In [8]:
def estimate_balance_temp(df: pd.DataFrame, target_col: str, temp_col: str, candidates=None) -> float:
    """
    Grid-searches the temperature bin with the lowest average demand -- an inspectable first
    pass for YOUR grid's balance temperature. For a more rigorous estimate, fit a piecewise-linear
    (segmented) regression of demand on temperature and take the breakpoint. Always sanity-check
    against your own scatter plot before trusting this.
    """
    if candidates is None:
        lo, hi = df[temp_col].quantile(0.02), df[temp_col].quantile(0.98)
        candidates = np.linspace(lo, hi, 40)
    bins = pd.cut(df[temp_col], bins=candidates)
    means = df.groupby(bins, observed=True)[target_col].mean()
    best_bin = means.idxmin()
    return float(best_bin.mid)


def chronological_split(dataset: Dataset, train_frac=0.8, val_frac=0.1):
    """Time-ordered split -- never shuffle before this. Returns (train_ds, val_ds, test_ds)."""
    n = len(dataset)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_ds = torch.utils.data.Subset(dataset, range(0, n_train))
    val_ds = torch.utils.data.Subset(dataset, range(n_train, n_train + n_val))
    test_ds = torch.utils.data.Subset(dataset, range(n_train + n_val, n))
    return train_ds, val_ds, test_ds


@torch.no_grad()
def evaluate_real_units(model, loader, target_scaler, device):
    """MAPE / RMSE computed after inverse-transforming back to original demand units."""
    model.eval()
    all_preds, all_true = [], []
    for past_target, covariates, future_target in loader:
        past_target = past_target.to(device)
        covariates = covariates.to(device)
        preds = model(past_target, covariates)
        all_preds.append(preds.cpu().numpy())
        all_true.append(future_target.numpy())
    preds = np.concatenate(all_preds, axis=0).reshape(-1, 1)
    true = np.concatenate(all_true, axis=0).reshape(-1, 1)
    preds_real = target_scaler.inverse_transform(preds).ravel()
    true_real = target_scaler.inverse_transform(true).ravel()
    mape = float(np.mean(np.abs((true_real - preds_real) / (true_real + 1e-6))) * 100)
    rmse = float(np.sqrt(np.mean((true_real - preds_real) ** 2)))
    return {"MAPE_%": mape, "RMSE": rmse}

## 6. Training loop

In [9]:
def train_pi_tide(
    df: pd.DataFrame,
    target_col: str = "demand",
    temp_col: str = "temp",
    covariate_cols: list = None,
    lookback: int = 168,
    horizon: int = 24,
    balance_temp: float = None,
    batch_size: int = 64,
    epochs: int = 100,
    lr: float = 1e-3,
    # --- Table 7 architecture hyper-parameters (wired for hypertuning) ---
    hidden_dim: int = 128,              # hiddenSize in [256, 512, 1024]
    encoder_layers: int = 2,            # numEncoderLayers in [1, 2, 3]
    decoder_layers: int = 2,            # numDecoderLayers in [1, 2, 3]
    decoder_output_dim: int = 16,       # decoderOutputDim in [4, 8, 16, 32]
    temporal_decoder_hidden: int = None,  # temporalDecoderHidden in [32, 64, 128]; None -> hidden_dim (backward-compat)
    dropout: float = 0.1,               # dropoutLevel in [0.0, 0.1, 0.2, 0.3, 0.5]
    use_layer_norm: bool = True,        # layerNorm in [True, False]
    use_revin: bool = False,            # revIn in [True, False]
    use_physics: bool = True,
    lambda_sens: float = 0.1,
    lambda_nonneg: float = 0.1,
    lambda_ramp: float = 0.1,
    lambda_env: float = 0.1,            # temperature-power envelope weight
    cooling_only: bool = False,         # True -> expected sensitivity sign is +1 everywhere
    physical_max_ramp: float = None,    # MW/h from grid specs (overrides data percentile)
    patience: int = 5,
    checkpoint_path: str = "pi_tide_best.pt",
    device: str = device,
    verbose: bool = True,
    return_val_loss: bool = False,
):
    """
    use_physics=False turns off all three physics terms (pure data-loss TiDE) -- run it both ways
    on the SAME data/split/seed as a baseline-vs-PI-TiDE ablation.

    If return_val_loss=True, returns (model, target_scaler, cov_scaler, test_metrics, best_val_loss)
    so the hypertuner can select on validation MSE. Otherwise returns the original 4-tuple.
    """
    if covariate_cols is None:
        covariate_cols = [temp_col]
    temp_idx = covariate_cols.index(temp_col)

    if balance_temp is None:
        balance_temp = estimate_balance_temp_piecewise(df, target_col, temp_col)
        if verbose:
            print(f"Estimated balance_temp (piecewise): {balance_temp:.2f} (verify against your own scatter plot)")

    df = df.copy()
    n_total = len(df)
    n_train_rows = int(0.8 * n_total)

    target_scaler = StandardScaler().fit(df[[target_col]].iloc[:n_train_rows])
    cov_scaler = StandardScaler().fit(df[covariate_cols].iloc[:n_train_rows])
    df[target_col] = target_scaler.transform(df[[target_col]])
    df[covariate_cols] = cov_scaler.transform(df[covariate_cols])

    temp_mean = cov_scaler.mean_[temp_idx]
    temp_std = cov_scaler.scale_[temp_idx]
    scaled_balance_temp = (balance_temp - temp_mean) / temp_std

    dataset = DemandWindowDataset(df, target_col, covariate_cols, lookback, horizon)
    train_ds, val_ds, test_ds = chronological_split(dataset, train_frac=0.8, val_frac=0.1)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # Physical max ramp (MW/h) -> scaled
    if physical_max_ramp is not None:
        max_ramp = physical_max_ramp / target_scaler.scale_[0]
    else:
        raw_train_target = df[target_col].values[:n_train_rows]
        max_ramp = float(np.percentile(np.abs(np.diff(raw_train_target)), 99))

    model = PITiDE(
        lookback=lookback,
        horizon=horizon,
        n_covariates=len(covariate_cols),
        temp_idx=temp_idx,
        hidden_dim=hidden_dim,
        encoder_layers=encoder_layers,
        decoder_layers=decoder_layers,
        decoder_output_dim=decoder_output_dim,
        temporal_decoder_hidden=temporal_decoder_hidden,
        dropout=dropout,
        use_layer_norm=use_layer_norm,
        use_revin=use_revin,
    ).to(device)

    physics = DegreeDayPhysics(balance_temp=scaled_balance_temp, cooling_only=cooling_only)
    # Fit temperature-power envelope on training data (before scaling)
    train_df = df.iloc[:n_train_rows].copy()
    # Inverse transform target
    train_df[target_col] = target_scaler.inverse_transform(train_df[[target_col]]).ravel()
    # Inverse transform temp (single column from covariates)
    temp_scaled = train_df[[temp_col]].values
    temp_orig = cov_scaler.inverse_transform(
        np.hstack([temp_scaled, np.zeros((len(temp_scaled), len(covariate_cols)-1))])
    )[:, 0]
    train_df[temp_col] = temp_orig
    physics.fit_envelope(train_df, temp_col=temp_col, target_col=target_col)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_data_loss, epoch_phys_loss = 0.0, 0.0
        for past_target, covariates, future_target in train_loader:
            past_target = past_target.to(device)
            covariates = covariates.to(device)
            future_target = future_target.to(device)

            optimizer.zero_grad()
            if use_physics:
                covariates.requires_grad_(True)
            preds = model(past_target, covariates)

            data_loss = F.mse_loss(preds, future_target)
            if use_physics:
                phys_loss, _ = physics_informed_loss(
                    model, past_target, covariates, preds, physics, max_ramp,
                    lambda_sens=lambda_sens, lambda_nonneg=lambda_nonneg,
                    lambda_ramp=lambda_ramp, lambda_env=lambda_env,
                )
            else:
                phys_loss = torch.tensor(0.0, device=device)
            loss = data_loss + phys_loss
            loss.backward()
            optimizer.step()

            epoch_data_loss += data_loss.item() * past_target.size(0)
            epoch_phys_loss += float(phys_loss.detach()) * past_target.size(0)

        epoch_data_loss /= len(train_ds)
        epoch_phys_loss /= len(train_ds)

        model.eval()
        val_loss_total, violation_count, violation_total = 0.0, 0, 0
        cold_viol, hot_viol = 0, 0
        cold_total, hot_total = 0, 0
        for past_target, covariates, future_target in val_loader:
            past_target = past_target.to(device)
            covariates = covariates.to(device).requires_grad_(True)
            future_target = future_target.to(device)
            preds = model(past_target, covariates)
            val_loss_total += F.mse_loss(preds, future_target).item() * past_target.size(0)

            if use_physics:
                grad_outputs = torch.ones_like(preds)
                d_pred_d_cov = torch.autograd.grad(preds, covariates, grad_outputs=grad_outputs, retain_graph=False)[0]
                d_pred_d_temp = d_pred_d_cov[:, lookback:, temp_idx]
                future_temp = covariates[:, lookback:, temp_idx]
                expected_sign = physics.expected_sensitivity_sign(future_temp)
                violations = (expected_sign * d_pred_d_temp < 0).float()
                violation_count += violations.sum().item()
                violation_total += violations.numel()

                # Per-regime violations
                cold_mask = future_temp < physics.T_b
                hot_mask = future_temp >= physics.T_b
                if cold_mask.any():
                    cold_viol += violations[cold_mask].sum().item()
                    cold_total += cold_mask.sum().item()
                if hot_mask.any():
                    hot_viol += violations[hot_mask].sum().item()
                    hot_total += hot_mask.sum().item()

        val_loss = val_loss_total / len(val_ds)
        violation_rate = violation_count / max(violation_total, 1) if use_physics else float("nan")
        cold_viol_rate = cold_viol / max(cold_total, 1) if use_physics else float("nan")
        hot_viol_rate = hot_viol / max(hot_total, 1) if use_physics else float("nan")

        if verbose:
            print(
                f"epoch {epoch:3d} | data_loss {epoch_data_loss:.4f} | phys_loss {epoch_phys_loss:.4f} "
                f"| val_loss {val_loss:.4f} | viol_rate {violation_rate:.3f} "
                f"| cold_viol {cold_viol_rate:.3f} hot_viol {hot_viol_rate:.3f}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({"model_state": model.state_dict(), "config": {
                "lookback": lookback, "horizon": horizon,
                "n_covariates": len(covariate_cols), "temp_idx": temp_idx,
                "hidden_dim": hidden_dim, "encoder_layers": encoder_layers,
                "decoder_layers": decoder_layers, "decoder_output_dim": decoder_output_dim,
                "temporal_decoder_hidden": temporal_decoder_hidden, "dropout": dropout,
                "use_layer_norm": use_layer_norm, "use_revin": use_revin, "lr": lr,
            }}, checkpoint_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (no val improvement for {patience} epochs).")
                break

    model.load_state_dict(torch.load(checkpoint_path, map_location=device)["model_state"])
    test_metrics = evaluate_real_units(model, test_loader, target_scaler, device)
    if verbose:
        print(f"Test set (original units): {test_metrics}")

    if return_val_loss:
        return model, target_scaler, cov_scaler, test_metrics, best_val_loss
    return model, target_scaler, cov_scaler, test_metrics


## 7. Load your data

Your dataset's columns:

```
DateTime, temperature, humidity, surface_pressure, Generation, Demand
```

Mapped as: `target_col="Demand"`, `temp_col="temperature"`, and `covariate_cols=["temperature", "humidity", "surface_pressure"]`.

**`Generation` is deliberately left out of the covariates.** TiDE's future-covariate slot assumes
values are *known ahead of time* for the forecast horizon (like a weather forecast or a calendar
flag). Generation is itself a grid outcome, not an exogenous driver -- you generally wouldn't know
future generation at the moment you're forecasting future demand, so including it as a "future"
covariate would leak information a real deployment wouldn't have. If you specifically want to study
demand-generation coupling (e.g. residual/net-load forecasting), that's a different, deliberate
setup -- happy to build that separately if it's what you want, but I did not assume it here.

The synthetic fallback only exists so this notebook runs standalone for a smoke test -- **do not use
it for real results.**

In [10]:
USE_REAL_DATA = True  # flip to True once the path below points to your actual file

if USE_REAL_DATA:
    df = pd.read_csv(r"E:\Machine Learning Research\Physics Informed TiDE\Kaggle_input_BangladeshData_2016_2024.csv", parse_dates=["DateTime"])
    df = df.sort_values("DateTime").reset_index(drop=True)
    df = df.rename(columns={"Demand": "demand", "temperature": "temp"})
    # humidity and surface_pressure keep their original names
    covariate_cols = ["temp", "humidity", "surface_pressure"]
    # Generation is intentionally excluded -- see markdown note above
else:
    rng = np.random.default_rng(0)
    n = 2000
    t = np.arange(n)
    temp = 20 + 10 * np.sin(2 * np.pi * t / (24 * 30)) + rng.normal(0, 1, n)
    balance = 22.0
    demand = (
        500
        + 15 * np.clip(balance - temp, 0, None)
        + 20 * np.clip(temp - balance, 0, None)
        + 30 * np.sin(2 * np.pi * t / 24)
        + rng.normal(0, 5, n)
    )
    humidity = 50 + 10 * np.sin(2 * np.pi * t / (24 * 7)) + rng.normal(0, 3, n)
    surface_pressure = 1013 + rng.normal(0, 3, n)
    df = pd.DataFrame({"demand": demand, "temp": temp, "humidity": humidity, "surface_pressure": surface_pressure})
    covariate_cols = ["temp", "humidity", "surface_pressure"]

df.head()

,DateTime,temp,humidity,surface_pressure,Generation,demand
0,2016-01-01 00:00:00,17.51,69.47,101.70,4211.0,4211.0
1,2016-01-01 01:00:00,16.72,72.76,101.67,4140.0,4140.0
2,2016-01-01 02:00:00,15.95,76.12,101.64,3941.0,3941.0
3,2016-01-01 03:00:00,15.36,78.96,101.63,3724.0,3724.0
4,2016-01-01 04:00:00,14.95,81.18,101.65,3603.0,3603.0


## 8. Baseline-vs-physics ablation
Runs the same seed/lookback/horizon with `use_physics=False` (plain TiDE) and `use_physics=True`
(PI-TiDE), so the comparison that actually matters for the paper — does the physics loss help —
is visible directly.

In [ ]:
common_kwargs = dict(
    target_col="demand",
    temp_col="temp",
    covariate_cols=covariate_cols,
    lookback=72,     # match your existing GA-TiDE benchmark lookback when running on real data
    horizon=24,      # match your existing GA-TiDE benchmark horizon
    epochs=30,       # raise to 30-100 on real data / GPU
    patience=5,
    batch_size=512,
    lr=1e-3,
    hidden_dim=64,
    encoder_layers=1,
    decoder_layers=1,
    decoder_output_dim=8,
    temporal_decoder_hidden=32,
    dropout=0.1,
    use_layer_norm=True,
    use_revin=False,
    physical_max_ramp=500.0,  # MW/h from grid specs
)

print("=== Baseline TiDE (physics OFF) ===")
torch.manual_seed(0)
_, _, _, baseline_metrics = train_pi_tide(
    df, use_physics=False, checkpoint_path="baseline_tide.pt", **common_kwargs
)

print("\n=== PI-TiDE (physics ON: cooling_only + envelope) ===")
torch.manual_seed(0)
_, _, _, pi_metrics = train_pi_tide(
    df,
    use_physics=True,
    lambda_sens=0.02,
    lambda_nonneg=0.01,
    lambda_ramp=0.01,
    lambda_env=0.02,
    cooling_only=True,        # Bangladesh: monotonic demand vs temp
    checkpoint_path="pi_tide_best.pt",
    **common_kwargs
)


=== Baseline TiDE (physics OFF) ===
Estimated balance_temp (piecewise): 25.47 (verify against your own scatter plot)
epoch   1 | data_loss 0.3068 | phys_loss 0.0000 | val_loss 0.3308 | viol_rate nan | cold_viol nan hot_viol nan
epoch   2 | data_loss 0.1373 | phys_loss 0.0000 | val_loss 0.2850 | viol_rate nan | cold_viol nan hot_viol nan
epoch   3 | data_loss 0.1174 | phys_loss 0.0000 | val_loss 0.2667 | viol_rate nan | cold_viol nan hot_viol nan
epoch   4 | data_loss 0.1090 | phys_loss 0.0000 | val_loss 0.2602 | viol_rate nan | cold_viol nan hot_viol nan
epoch   5 | data_loss 0.1044 | phys_loss 0.0000 | val_loss 0.2535 | viol_rate nan | cold_viol nan hot_viol nan
epoch   6 | data_loss 0.1013 | phys_loss 0.0000 | val_loss 0.2501 | viol_rate nan | cold_viol nan hot_viol nan
epoch   7 | data_loss 0.0991 | phys_loss 0.0000 | val_loss 0.2467 | viol_rate nan | cold_viol nan hot_viol nan
epoch   8 | data_loss 0.0974 | phys_loss 0.0000 | val_loss 0.2443 | viol_rate nan | cold_viol nan hot_viol

C:\Users\User\AppData\Local\Temp\ipykernel_18108\2066888097.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_locat

Test set (original units): {'MAPE_%': 8.080214262008667, 'RMSE': 1025.6611328125}

=== PI-TiDE (physics ON: cooling_only + envelope) ===
Estimated balance_temp (piecewise): 25.47 (verify against your own scatter plot)
epoch   1 | data_loss 0.3064 | phys_loss 0.0036 | val_loss 0.3306 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   2 | data_loss 0.1372 | phys_loss 0.0036 | val_loss 0.2849 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   3 | data_loss 0.1174 | phys_loss 0.0037 | val_loss 0.2665 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   4 | data_loss 0.1090 | phys_loss 0.0037 | val_loss 0.2602 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   5 | data_loss 0.1044 | phys_loss 0.0038 | val_loss 0.2533 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   6 | data_loss 0.1013 | phys_loss 0.0038 | val_loss 0.2498 | viol_rate 0.000 | cold_viol 0.000 hot_viol 0.000
epoch   7 | data_loss 0.0991 | phys_loss 0.0038 | val_loss 0.2464 | viol_rate 0.

In [ ]:
print("=== PI-TiDE (physics ON) ===")
torch.manual_seed(0)
_, _, _, pi_metrics = train_pi_tide(
    df, use_physics=True, checkpoint_path="pi_tide_best.pt", **common_kwargs
)

=== PI-TiDE (physics ON) ===
Estimated balance_temp from data: 22.18 (verify against your own scatter plot)
epoch   1 | data_loss 0.7037 | phys_loss 0.0176 | val_loss 0.5560 | physical_violation_rate 0.000
epoch   2 | data_loss 0.3372 | phys_loss 0.0253 | val_loss 0.3510 | physical_violation_rate 0.000
epoch   3 | data_loss 0.2161 | phys_loss 0.0304 | val_loss 0.2723 | physical_violation_rate 0.000
epoch   4 | data_loss 0.1740 | phys_loss 0.0347 | val_loss 0.2409 | physical_violation_rate 0.000
epoch   5 | data_loss 0.1579 | phys_loss 0.0371 | val_loss 0.2293 | physical_violation_rate 0.000
epoch   6 | data_loss 0.1520 | phys_loss 0.0376 | val_loss 0.2243 | physical_violation_rate 0.000
epoch   7 | data_loss 0.1480 | phys_loss 0.0384 | val_loss 0.2196 | physical_violation_rate 0.000
epoch   8 | data_loss 0.1458 | phys_loss 0.0384 | val_loss 0.2167 | physical_violation_rate 0.000
epoch   9 | data_loss 0.1437 | phys_loss 0.0386 | val_loss 0.2148 | physical_violation_rate 0.000
epoch  10 

C:\Users\User\AppData\Local\Temp\ipykernel_18120\4232964506.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_locat

In [ ]:
print("Baseline test metrics:", baseline_metrics)
print("PI-TiDE  test metrics:", pi_metrics)

Baseline test metrics: {'MAPE_%': 2.43326835334301, 'RMSE': 18.951478958129883}
PI-TiDE  test metrics: {'MAPE_%': 2.3566847667098045, 'RMSE': 18.344175338745117}


In [ ]:
# Systematic ablation study: isolate each physics component
# Mirrors manuscript Table 5

print("=== Systematic Physics Ablation ===")

common_ablation = dict(
    target_col="demand",
    temp_col="temp",
    covariate_cols=covariate_cols,
    lookback=72,
    horizon=24,
    balance_temp=None,
    batch_size=512,
    epochs=15,
    patience=3,
    lr=1e-3,
    hidden_dim=64,
    encoder_layers=1,
    decoder_layers=1,
    decoder_output_dim=8,
    temporal_decoder_hidden=32,
    dropout=0.1,
    use_layer_norm=True,
    use_revin=False,
    physical_max_ramp=500.0,
    cooling_only=True,
)

ablation_configs = [
    ("TiDE_no_physics",          dict(use_physics=False, lambda_sens=0, lambda_nonneg=0, lambda_ramp=0, lambda_env=0)),
    ("Ramp_only",                dict(use_physics=True,  lambda_sens=0, lambda_nonneg=0, lambda_ramp=0.01, lambda_env=0)),
    ("Nonneg_only",              dict(use_physics=True,  lambda_sens=0, lambda_nonneg=0.01, lambda_ramp=0, lambda_env=0)),
    ("Envelope_only",            dict(use_physics=True,  lambda_sens=0, lambda_nonneg=0, lambda_ramp=0, lambda_env=0.02)),
    ("All_static_lambda",        dict(use_physics=True,  lambda_sens=0.1, lambda_nonneg=0.1, lambda_ramp=0.1, lambda_env=0.1)),
    ("PI_TiDE_dynamic_lambda",   dict(use_physics=True,  lambda_sens=0.02, lambda_nonneg=0.01, lambda_ramp=0.01, lambda_env=0.02)),
]

import pandas as pd
ablation_results = []

for name, phys_kwargs in ablation_configs:
    print(f"--- {name} ---")
    torch.manual_seed(0)
    np.random.seed(0)
    _, _, _, test_metrics, val_loss = train_pi_tide(
        df,
        checkpoint_path=f"ablation_{name}.pt",
        verbose=False,
        return_val_loss=True,
        **common_ablation,
        **phys_kwargs
    )
    ablation_results.append({
        "Configuration": name,
        "MAPE_%": f"{test_metrics['MAPE_%']:.2f}",
        "RMSE": f"{test_metrics['RMSE']:.2f}",
        "Val_Loss": f"{val_loss:.4f}",
    })
    print(f"  MAPE: {test_metrics['MAPE_%']:.2f}%  RMSE: {test_metrics['RMSE']:.2f}  Val: {val_loss:.4f}")

ablation_df = pd.DataFrame(ablation_results)
print("\n=== Ablation Summary ===")
print(ablation_df.to_string(index=False))
ablation_df.to_csv("ablation_results.csv", index=False)
print("Saved to ablation_results.csv")


## 9. Hyperparameter tuning for BOTH models (Table 7 grid search)

Exhaustive **grid search** over the Table 7 ranges below, run **separately for both models**
on the SAME grid / split / seed so the comparison is fair:
- **Baseline TiDE** = plain data-loss TiDE (`use_physics=False`)
- **PI-TiDE** = physics-informed TiDE (`use_physics=True`)

`learningRate` is log-scale in `[1e-5, 1e-2]` and is discretized to a log-spaced grid
`[1e-5, 1e-4, 1e-3, 1e-2]` for the enumeration (edit `SEARCH_SPACE["learningRate"]` to use a
finer/coarser grid). Each grid point trains briefly (`trial_epochs`, early-stopped on validation
MSE) and the best config **per model** is selected on **validation loss** — test metrics are
reported for information only.

| Hyper-parameter | Grid |
|---|---|
| hiddenSize | [256, 512, 1024] |
| numEncoderLayers | [1, 2, 3] |
| numDecoderLayers | [1, 2, 3] |
| decoderOutputDim | [4, 8, 16, 32] |
| temporalDecoderHidden | [32, 64, 128] |
| dropoutLevel | [0.0, 0.1, 0.2, 0.3, 0.5] |
| layerNorm | [True, False] |
| learningRate | log-scale grid [1e-5, 1e-4, 1e-3, 1e-2] in [1e-5, 1e-2] |
| revIn | [True, False] |

Full grid size = 3 x 3 x 3 x 4 x 3 x 5 x 2 x 4 x 2 = 25,920 runs **per model**, which is
infeasible to run exhaustively. Use `max_trials` to cap each run (first-N grid points in
deterministic order) or pass a narrowed `search_space` override to `hypertune_pi_tide()`.

Mapping to code: `hiddenSize -> hidden_dim`, `numEncoderLayers -> encoder_layers`,
`numDecoderLayers -> decoder_layers`, `decoderOutputDim -> decoder_output_dim`,
`temporalDecoderHidden -> temporal_decoder_hidden`, `dropoutLevel -> dropout`,
`layerNorm -> use_layer_norm`, `learningRate -> lr`, `revIn -> use_revin`.


In [ ]:
import itertools

# Table 7: grid for exhaustive search (learningRate log-scale [1e-5, 1e-2] discretized)
SEARCH_SPACE = {
    "hiddenSize": [256, 512, 1024],
    "numEncoderLayers": [1, 2, 3],
    "numDecoderLayers": [1, 2, 3],
    "decoderOutputDim": [4, 8, 16, 32],
    "temporalDecoderHidden": [32, 64, 128],
    "dropoutLevel": [0.0, 0.1, 0.2, 0.3, 0.5],
    "layerNorm": [True, False],
    "learningRate": [1e-5, 1e-4, 1e-3, 1e-2],  # log-spaced grid in [1e-5, 1e-2]
    "revIn": [True, False],
}

GRID_KEYS = ["hiddenSize", "numEncoderLayers", "numDecoderLayers", "decoderOutputDim",
             "temporalDecoderHidden", "dropoutLevel", "layerNorm", "learningRate", "revIn"]

def build_grid(search_space=None) -> list:
    """Cartesian product of the Table 7 grid as a list of config dicts (deterministic order)."""
    ss = search_space if search_space is not None else SEARCH_SPACE
    keys = [k for k in GRID_KEYS if k in ss]
    return [dict(zip(keys, combo)) for combo in itertools.product(*(ss[k] for k in keys))]

def grid_size(search_space=None) -> int:
    ss = search_space if search_space is not None else SEARCH_SPACE
    n = 1
    for k in GRID_KEYS:
        if k in ss:
            n *= len(ss[k])
    return n

print(f"Full Table 7 grid size: {grid_size()} combos")
build_grid()[:2]


In [ ]:
def hypertune_pi_tide(
    df: pd.DataFrame,
    target_col: str = "demand",
    temp_col: str = "temp",
    covariate_cols: list = None,
    lookback: int = 72,
    horizon: int = 24,
    balance_temp: float = None,
    search_space: dict = None,
    max_trials: int = None,
    trial_epochs: int = 15,
    trial_patience: int = 3,
    batch_size: int = 64,
    use_physics: bool = True,
    lambda_sens: float = 0.1,
    lambda_nonneg: float = 0.1,
    lambda_ramp: float = 0.1,
    seed: int = 0,
    checkpoint_dir: str = "hypertune_ckpts",
    device: str = device,
    verbose: bool = True,
):
    """
    Exhaustive grid search over Table 7 (SEARCH_SPACE / build_grid) for ONE model.
    Call it twice -- once with use_physics=False (plain baseline TiDE) and once with
    use_physics=True (PI-TiDE) -- for a fair both-models comparison (see run cell).

    - Grid order is deterministic (itertools.product over GRID_KEYS).
    - Pass search_space={...} (subset of SEARCH_SPACE keys) to narrow the grid.
    - Pass max_trials=N to run only the first N grid points (useful: full grid is 25,920 runs).
    - Each trial calls train_pi_tide() briefly with return_val_loss=True; best config
      (lowest val MSE) is returned with the full ranked results dataframe.
      Test metrics are logged per trial for information only -- selection uses val_loss.
    """
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)
    if covariate_cols is None:
        covariate_cols = [temp_col]
    if balance_temp is None:
        balance_temp = estimate_balance_temp(df, target_col, temp_col)
        if verbose:
            print(f"[hypertune] fixed balance_temp={balance_temp:.2f} for all trials (estimated once)")

    grid = build_grid(search_space)
    total = len(grid)
    if max_trials is not None:
        grid = grid[:max_trials]
    if verbose:
        print(f"[hypertune] grid search: running {len(grid)}/{total} grid points")

    records = []
    for trial, cfg in enumerate(grid):
        torch.manual_seed(seed + trial)
        np.random.seed(seed + trial)
        ckpt = os.path.join(checkpoint_dir, f"trial_{trial:03d}.pt")
        _, _, _, test_metrics, val_loss = train_pi_tide(
            df,
            target_col=target_col, temp_col=temp_col, covariate_cols=covariate_cols,
            lookback=lookback, horizon=horizon, balance_temp=balance_temp,
            batch_size=batch_size, epochs=trial_epochs, patience=trial_patience,
            lr=cfg["learningRate"],
            hidden_dim=cfg["hiddenSize"],
            encoder_layers=cfg["numEncoderLayers"],
            decoder_layers=cfg["numDecoderLayers"],
            decoder_output_dim=cfg["decoderOutputDim"],
            temporal_decoder_hidden=cfg["temporalDecoderHidden"],
            dropout=cfg["dropoutLevel"],
            use_layer_norm=cfg["layerNorm"],
            use_revin=cfg["revIn"],
            use_physics=use_physics,
            lambda_sens=lambda_sens, lambda_nonneg=lambda_nonneg, lambda_ramp=lambda_ramp,
            checkpoint_path=ckpt, device=device, verbose=False, return_val_loss=True,
        )
        rec = {
            **cfg,
            "val_loss": float(val_loss),
            "test_MAPE": test_metrics["MAPE_%"],
            "test_RMSE": test_metrics["RMSE"],
            "checkpoint": ckpt,
        }
        records.append(rec)
        if verbose:
            print(
                f"trial {trial + 1}/{len(grid)} | val {val_loss:.4f} | "
                f"MAPE {test_metrics['MAPE_%']:.2f}% RMSE {test_metrics['RMSE']:.2f} | {cfg}"
            )

    results_df = pd.DataFrame(records).sort_values("val_loss", ignore_index=True)
    best = results_df.iloc[0].to_dict()
    if verbose:
        print("\nBest config (lowest val_loss):")
        print(best)
    return best, results_df


In [ ]:
# --- Table 7 grid search for BOTH models (same grid / split / seed -> fair comparison) ---
# Full grid = 25,920 runs per model: cap with max_trials and/or narrow search_space.
hypertune_kwargs = dict(
    target_col="demand",
    temp_col="temp",
    covariate_cols=covariate_cols,
    lookback=72,      # keep consistent with the baseline ablation above
    horizon=24,
    max_trials=20,    # first 20 grid points per model; None = exhaustive, or pass search_space={...}
    trial_epochs=15,  # cheap screening budget per trial
    trial_patience=3,
    seed=0,
)

print("=== Baseline TiDE hypertune (physics OFF) ===")
best_baseline, results_baseline = hypertune_pi_tide(
    df, use_physics=False, checkpoint_dir="hypertune_ckpts_baseline", **hypertune_kwargs
)
print("\n=== PI-TiDE hypertune (physics ON) ===")
best_pi, results_pi = hypertune_pi_tide(
    df, use_physics=True, checkpoint_dir="hypertune_ckpts_pi", **hypertune_kwargs
)

print("\nBaseline top-5:")
print(results_baseline.head(5).to_string())
print("\nPI-TiDE top-5:")
print(results_pi.head(5).to_string())


In [ ]:
# Full retrain of BOTH hypertune winners (raise epochs/patience for the final models)
def _retrain_from_best(best_cfg, use_physics, checkpoint_path):
    torch.manual_seed(0)
    return train_pi_tide(
        df,
        target_col="demand",
        temp_col="temp",
        covariate_cols=covariate_cols,
        lookback=72,
        horizon=24,
        lr=best_cfg["learningRate"],
        hidden_dim=int(best_cfg["hiddenSize"]),
        encoder_layers=int(best_cfg["numEncoderLayers"]),
        decoder_layers=int(best_cfg["numDecoderLayers"]),
        decoder_output_dim=int(best_cfg["decoderOutputDim"]),
        temporal_decoder_hidden=int(best_cfg["temporalDecoderHidden"]),
        dropout=float(best_cfg["dropoutLevel"]),
        use_layer_norm=bool(best_cfg["layerNorm"]),
        use_revin=bool(best_cfg["revIn"]),
        use_physics=use_physics,
        epochs=50,
        patience=5,
        checkpoint_path=checkpoint_path,
        return_val_loss=True,
    )

print("=== Retrain best BASELINE TiDE ===")
baseline_model, _, _, baseline_ht_metrics, baseline_ht_val = _retrain_from_best(
    best_baseline, use_physics=False, checkpoint_path="baseline_tide_hypertuned_best.pt"
)
print("=== Retrain best PI-TiDE ===")
pi_model, _, _, pi_ht_metrics, pi_ht_val = _retrain_from_best(
    best_pi, use_physics=True, checkpoint_path="pi_tide_hypertuned_best.pt"
)

print(f"Hypertuned baseline test: {baseline_ht_metrics} (val {baseline_ht_val:.4f})")
print(f"Hypertuned PI-TiDE  test: {pi_ht_metrics} (val {pi_ht_val:.4f})")


## 11. Next steps
- Probabilistic PI-TiDE: quantile regression with physical bounds on prediction intervals
- Multi-grid transfer: static covariate encoders for cross-station generalization
- Hard constraints via projection layers for strict feasibility guarantees
- Real-time deployment: integrate with SCADA/EMS for closed-loop validation
